# NVIDIA Earth-2 Weather Simulation

This notebook demonstrates how to use the Earth-2 simulation framework for AI-powered weather forecasting.

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from earth2_simulation import Earth2Simulator
from ensemble_forecast import EnsembleForecast
from visualize import ForecastVisualizer
import xarray as xr
import matplotlib.pyplot as plt

print("✓ Imports successful")

## 1. Run a Single Model Forecast

In [ ]:
# Initialize simulator
simulator = Earth2Simulator(
    start_date="2025-01-01T00:00:00",
    steps=8
)

# Run FourCastNet model
output_path = simulator.run_fourcastnet()
print(f"Forecast saved to: {output_path}")

## 2. Load and Inspect Forecast Data

In [ ]:
# Load forecast
forecast = simulator.load_forecast(output_path)

print("Forecast dimensions:")
print(forecast.dims)
print("\nAvailable variables:")
print(list(forecast.data_vars))
print("\nTime steps:")
print(forecast.time.values)

## 3. Visualize Temperature Field

In [ ]:
viz = ForecastVisualizer(output_path)
viz.plot_temperature(time_step=0)

## 4. Visualize Wind Field

In [ ]:
viz.plot_wind(time_step=0)

## 5. Analyze Temperature Evolution

In [ ]:
if 't2m' in forecast:
    # Compute global mean temperature
    global_mean = forecast['t2m'].mean(dim=['lat', 'lon']) - 273.15  # Convert to Celsius
    
    # Plot evolution
    plt.figure(figsize=(10, 5))
    plt.plot(range(len(global_mean)), global_mean.values, marker='o')
    plt.xlabel('Time Step (6-hour intervals)')
    plt.ylabel('Global Mean Temperature (°C)')
    plt.title('Global Mean Temperature Evolution')
    plt.grid(True)
    plt.show()

## 6. Multi-Model Ensemble (Optional)

In [ ]:
# Run multiple models
forecast_paths = simulator.run_all_models()

# Create ensemble
ensemble = EnsembleForecast(forecast_paths)

# Compute ensemble mean
ensemble_mean = ensemble.compute_ensemble_mean('t2m')
print(f"Ensemble mean shape: {ensemble_mean.shape}")

# Compute ensemble spread
ensemble_spread = ensemble.compute_ensemble_spread('t2m')
print(f"Ensemble spread shape: {ensemble_spread.shape}")

## 7. Custom Analysis

In [ ]:
# Example: Find maximum temperature location
if 't2m' in forecast:
    temp = forecast['t2m'].isel(time=0)
    max_temp_idx = temp.argmax(dim=['lat', 'lon'])
    max_temp = temp.max().values - 273.15
    
    print(f"Maximum temperature: {max_temp:.2f}°C")
    print(f"Location: Lat {temp.lat[max_temp_idx // len(temp.lon)].values:.2f}, "
          f"Lon {temp.lon[max_temp_idx % len(temp.lon)].values:.2f}")

## Summary

This notebook demonstrated:
- Running AI weather forecast models
- Loading and inspecting forecast data
- Creating visualizations
- Analyzing forecast outputs
- Creating multi-model ensembles

For more information, see the [Earth2Studio documentation](https://nvidia.github.io/earth2studio/).